In [1]:
import pandas as pd
import pathlib
import json
import gspread
from oauth2client.service_account import ServiceAccountCredentials


/home/joe/anaconda3/envs/bic/lib/python3.6/site-packages/requests/__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.18) or chardet (5.0.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)


In [2]:
def getXrefs():
    '''Reads the Inventory google sheet and gets the datasets title and cross-refs it to the Socrata 4x4 id.  Also
    gets the fields by 4x4 dataset id and by the title'''
    scope = ['https://www.googleapis.com/auth/spreadsheets.readonly',
             "https://www.googleapis.com/auth/drive.file",
                  "https://www.googleapis.com/auth/drive"]

    creds = ServiceAccountCredentials.from_json_keyfile_name('/home/joe/work/client_secret.json',
     scope)
    client = gspread.authorize(creds)

    gc = gspread.service_account("/home/joe/work/client_secret.json")
    # for gg in gc.list_spreadsheet_files():
    #      print("GGGGG ",gg)
    
    tracker = client.open('BIC Dataset Tracker').worksheet(
    'PublishedData')
   

    df = pd.DataFrame(tracker.get_all_records(head=2))

    return df
            
df = getXrefs()

In [3]:

xrefs=dict(zip(df['Dataset Title'], df['Socrata Link']))

In [4]:
desktop = pathlib.Path("/home/joe/bic_etl")
runEtls = []

files = list(desktop.rglob("*"))
# Which you can wrap in a list() constructor to materialize
for ff in files:
        if ("cdos" in str(ff) and str(ff).split("/")[-1] == "run_etl.json"):     
            runEtls.append(ff)

In [10]:
info = {}
info['group']=[]
info['title']=[]
info['4x4']=[]
info['infile']=[]
info['trans']=[]

for file in runEtls:
    print(file)
    fin=open(file)
    all=json.load(fin)
    file=str(file)
    start=file.find("cdos")
    end=file.find("run_etl")
    group=file[23:end-1]
    for tmp in all:
        title = tmp['title']
        prog=""
        infile=""
        trans = ""
        progTrans = ""
        if 'extract' in tmp:
            extra=tmp['extract']
            if 'file' in extra:
                prog = extra['file']
            if 'options' in extra:
                options = extra['options']
                for opt in options:
                    mm=opt.find("-f")
                    if mm > -1:
                       infile = opt[mm+2:].split("/")[-1]
        if 'transform' in tmp:
            trans = tmp['transform']
            if 'file' in trans:
                progTrans = trans['file'].split("/")[-1]
            
        info['group'].append(group)
        info['title'].append(title)
        info['infile'].append(infile)
        info['trans'].append(progTrans)
        info['4x4'].append(xrefs[title])
        

    

/home/joe/bic_etl/cdos/health/run_etl.json
/home/joe/bic_etl/cdos/government/run_etl.json
/home/joe/bic_etl/cdos/business/ucc/run_etl.json
/home/joe/bic_etl/cdos/business/business/run_etl.json
/home/joe/bic_etl/cdos/business/nonprofit/run_etl.json
/home/joe/bic_etl/cdos/lobbyist/run_etl.json


In [12]:
dfInfo=pd.DataFrame(info)

In [13]:
dfInfo.to_csv("datasets.csv",index=False)